# Assignment-3
Name: Avishkar Dinesh Chothwe

PRN: 22310789

Roll NO: 382028

In [1]:
!pip install transformers datasets torch accelerate

In [2]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

In [3]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
dataset = load_dataset(
    "text",
    data_files={"train": "stories.txt"}
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 19
    })
})


In [10]:
def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/19 [00:00<?, ? examples/s]

In [7]:
training_args = TrainingArguments(
    output_dir="./story_generator",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

In [14]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=30, training_loss=2.466888936360677, metrics={'train_runtime': 202.794, 'train_samples_per_second': 0.281, 'train_steps_per_second': 0.148, 'total_flos': 3723411456000.0, 'train_loss': 2.466888936360677, 'epoch': 3.0})

In [15]:
trainer.save_model("./story_generator_model")
tokenizer.save_pretrained("./story_generator_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./story_generator_model/tokenizer_config.json',
 './story_generator_model/tokenizer.json')

In [17]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Set pad token
tokenizer.pad_token = tokenizer.eos_token

prompt = "Once upon a time"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True
)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

output = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_length=150,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

story = tokenizer.decode(output[0], skip_special_tokens=True)

print(story)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Once upon a time, she was a woman, a mother, and a woman's mother, and when that time comes, we are a race of people.

I went to church once and then I was dead.

I was the first of the African Americans on the street who had been arrested when I was 15 for the crimes of sexual assault.

I was the first black woman who was arrested by the police after I was 17 and found by police that I was having an abortion.

I went to church once and then I was dead.

I was the first woman who was arrested by the police after I was 15 for the crimes of sexual assault.

I was the first black woman who was arrested
